# 1. Learned fusion repeated-seed consolidation

This notebook aggregates three fresh, controlled seeds (17, 42 and 73) across the same five outer folds. It refuses incomplete, mismatched or duplicated outputs; recomputes metrics from participant-level predictions; measures participant-level variability across seeds; and reports conflict-aware uncertainty from exact pairwise-conflict outputs.


## 1.1. Exact experiment paths and analysis settings

All controlled seed results are read from the isolated seed-sensitivity experiment. The previously completed experiment remains untouched and is not mislabelled as an initialisation-controlled seed replicate.


In [ ]:
# ============================================================
# 1. Exact paths, seeds, folds, and output locations
# ============================================================

from pathlib import Path
import hashlib
import json
import os

import numpy as np
import pandas as pd

from IPython.display import display

from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    log_loss,
    precision_score,
    roc_auc_score,
)


try:
    from google.colab import drive
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")
except ImportError:
    pass


PROJECT_ROOT = Path("/content/drive/MyDrive/adni_mri")
MODEL_ROOT = PROJECT_ROOT / "models" / "3mt_tmc_evidential"

TASK_NAME = "mci_prognosis"
OUTER_FOLDS = (0, 1, 2, 3, 4)
SEEDS = (17, 42, 73)
EXPECTED_PARTICIPANTS = 544
CLASSIFICATION_THRESHOLD = 0.50

SEED_EXPERIMENT_NAME = "gated_cmt_learned_gate_md050_seed_sensitivity"

SEED_TASK_ROOT = (
    MODEL_ROOT
    / "experiments"
    / SEED_EXPERIMENT_NAME
    / TASK_NAME
)

AGGREGATION_ROOT = (
    MODEL_ROOT
    / "experiments"
    / SEED_EXPERIMENT_NAME
    / "seed_sensitivity_aggregation"
    / TASK_NAME
)

TABLE_DIR = AGGREGATION_ROOT / "tables"
POOLED_DIR = AGGREGATION_ROOT / "pooled_predictions"

for directory in [AGGREGATION_ROOT, TABLE_DIR, POOLED_DIR]:
    directory.mkdir(parents=True, exist_ok=True)


def run_root(seed, fold):
    return SEED_TASK_ROOT / f"seed_{seed}" / f"fold_{fold}"


def atomic_write_csv(dataframe, destination):
    temporary = destination.with_name(destination.name + ".tmp")
    dataframe.to_csv(temporary, index=False)
    os.replace(temporary, destination)


print("=" * 72)
print("SEED-SENSITIVITY CONSOLIDATION")
print("=" * 72)
print(f"\nControlled seed root:\n{SEED_TASK_ROOT}")
print(f"\nAggregation outputs:\n{AGGREGATION_ROOT}")


## 1.2. Strictly validate and load all 15 seed/fold runs

Every run requires its `_SUCCESS.json` marker and matching hashes. The experiment, task, fold, seed, participant count and test-time mask behaviour are checked before loading. Across all seeds, each fold must contain exactly the same participants and targets.


In [ ]:
# ============================================================
# 2. Validate every required artifact before aggregation
# ============================================================

REQUIRED_PREDICTION_COLUMNS = {
    "RID",
    "TARGET",
    "FINAL_P_pMCI",
    "FINAL_UNCERTAINTY",
    "W_3MT",
    "W_TMC",
    "THREE_MT_P_pMCI",
    "THREE_MT_UNCERTAINTY",
    "TMC_P_pMCI",
    "TMC_UNCERTAINTY",
    "ORIGINAL_MODALITY_COUNT",
    "EFFECTIVE_MODALITY_COUNT",
}

NEW_CONFLICT_COLUMNS = {
    "MAX_PAIRWISE_CONFLICT",
    "MEAN_TMC_CONFLICT",
    "CONFLICT_AWARE_UNCERTAINTY",
}


def sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


prediction_tables = []
metadata_rows = []

for seed in SEEDS:
    for fold in OUTER_FOLDS:
        root = run_root(seed, fold)
        paths = {
            "configuration": root / "history" / "training_configuration.json",
            "best_checkpoint": root / "checkpoints" / "best_validation_auc_checkpoint.pt",
            "history": root / "history" / "training_history.csv",
            "predictions": root / "predictions" / "test_predictions.csv",
            "metrics": root / "predictions" / "test_metrics.json",
            "summary": root / "predictions" / "test_metrics_summary.csv",
            "success": root / "_SUCCESS.json",
        }

        required = [
            "configuration",
            "best_checkpoint",
            "history",
            "predictions",
            "metrics",
            "summary",
        ]
        missing = [name for name in required if not paths[name].is_file()]
        if missing:
            raise FileNotFoundError(
                f"Seed {seed}, fold {fold} is incomplete; missing: {missing}. "
                f"Run root: {root}"
            )

        with open(paths["configuration"], "r", encoding="utf-8") as handle:
            configuration = json.load(handle)
        with open(paths["metrics"], "r", encoding="utf-8") as handle:
            metrics = json.load(handle)

        expected_experiment = SEED_EXPERIMENT_NAME

        checks = {
            "configuration experiment": configuration.get("experiment_name") == expected_experiment,
            "configuration task": configuration.get("task") == TASK_NAME,
            "configuration fold": int(configuration.get("fold", -1)) == fold,
            "configuration seed": int(configuration.get("random_seed", -1)) == seed,
            "metrics experiment": metrics.get("experiment_name") == expected_experiment,
            "metrics task": metrics.get("task") == TASK_NAME,
            "metrics fold": int(metrics.get("fold", -1)) == fold,
            "test-time mask": float(metrics.get("maximum_modality_count_difference", np.nan)) == 0.0,
        }
        failed = [name for name, passed in checks.items() if not passed]
        if failed:
            raise ValueError(
                f"Seed {seed}, fold {fold} failed identity checks: {failed}"
            )

        predictions = pd.read_csv(paths["predictions"])
        missing_columns = REQUIRED_PREDICTION_COLUMNS.difference(predictions.columns)
        if missing_columns:
            raise KeyError(
                f"Seed {seed}, fold {fold} predictions lack: {sorted(missing_columns)}"
            )

        missing_conflict = NEW_CONFLICT_COLUMNS.difference(predictions.columns)
        if missing_conflict:
            raise KeyError(
                f"Seed {seed}, fold {fold} lacks conflict outputs: "
                f"{sorted(missing_conflict)}"
            )

        bounded_columns = [
            "FINAL_P_pMCI",
            "FINAL_UNCERTAINTY",
            "W_3MT",
            "W_TMC",
            "THREE_MT_P_pMCI",
            "THREE_MT_UNCERTAINTY",
            "TMC_P_pMCI",
            "TMC_UNCERTAINTY",
            "MAX_PAIRWISE_CONFLICT",
            "MEAN_TMC_CONFLICT",
            "CONFLICT_AWARE_UNCERTAINTY",
        ]
        for column in bounded_columns:
            numeric = pd.to_numeric(predictions[column], errors="raise")
            if not numeric.between(0.0, 1.0, inclusive="both").all():
                raise ValueError(
                    f"Seed {seed}, fold {fold}: {column} is outside [0, 1]."
                )

        if not np.allclose(
            predictions["W_3MT"] + predictions["W_TMC"],
            1.0,
            atol=1e-6,
        ):
            raise ValueError(f"Seed {seed}, fold {fold}: gate weights do not sum to one.")

        recalculated_conflict_aware = (
            1.0
            - (1.0 - predictions["FINAL_UNCERTAINTY"])
            * (1.0 - predictions["MAX_PAIRWISE_CONFLICT"])
        )
        if not np.allclose(
            predictions["CONFLICT_AWARE_UNCERTAINTY"],
            recalculated_conflict_aware,
            atol=1e-7,
        ):
            raise ValueError(
                f"Seed {seed}, fold {fold}: conflict-aware uncertainty mismatch."
            )

        if not paths["success"].is_file():
            raise FileNotFoundError(
                f"Seed {seed}, fold {fold} has no _SUCCESS.json marker."
            )
        with open(paths["success"], "r", encoding="utf-8") as handle:
            success = json.load(handle)
        if success.get("status") != "complete":
            raise ValueError(f"Seed {seed}, fold {fold} is not marked complete.")
        if int(success.get("random_seed", -1)) != seed or int(success.get("fold", -1)) != fold:
            raise ValueError(f"Seed {seed}, fold {fold} completion identity mismatch.")
        if success.get("test_predictions_sha256") != sha256(paths["predictions"]):
            raise ValueError(f"Seed {seed}, fold {fold} prediction hash mismatch.")
        if success.get("test_metrics_sha256") != sha256(paths["metrics"]):
            raise ValueError(f"Seed {seed}, fold {fold} metrics hash mismatch.")

        predictions = predictions.copy()
        predictions.insert(0, "FOLD", fold)
        predictions.insert(0, "SEED", seed)

        if predictions["RID"].duplicated().any():
            raise ValueError(f"Seed {seed}, fold {fold} contains duplicated RIDs.")
        if len(predictions) != int(metrics["test_participants"]):
            raise ValueError(f"Seed {seed}, fold {fold} participant count mismatch.")
        if not np.allclose(
            predictions["ORIGINAL_MODALITY_COUNT"],
            predictions["EFFECTIVE_MODALITY_COUNT"],
        ):
            raise ValueError(f"Seed {seed}, fold {fold} used test-time modality dropout.")

        prediction_tables.append(predictions)
        metadata_rows.append(
            {
                "SEED": seed,
                "FOLD": fold,
                "EXPERIMENT": expected_experiment,
                "CHECKPOINT_EPOCH": int(metrics["checkpoint_epoch"]),
                "CHECKPOINT_VALIDATION_AUC": float(metrics["checkpoint_validation_auc"]),
                "TEST_PARTICIPANTS": int(metrics["test_participants"]),
                "RUN_ROOT": str(root),
            }
        )


all_predictions = pd.concat(prediction_tables, ignore_index=True)
run_metadata = pd.DataFrame(metadata_rows)


# Every seed must contain one complete 544-participant OOF cohort.
seed_counts = all_predictions.groupby("SEED").agg(
    ROWS=("RID", "size"),
    UNIQUE_RIDS=("RID", "nunique"),
)
if not (seed_counts["ROWS"] == EXPECTED_PARTICIPANTS).all():
    raise ValueError("At least one seed does not contain 544 OOF predictions.")
if not (seed_counts["UNIQUE_RIDS"] == EXPECTED_PARTICIPANTS).all():
    raise ValueError("At least one seed contains duplicated or missing OOF RIDs.")


# Fold membership and targets must be identical across seeds.
identity = all_predictions[["SEED", "FOLD", "RID", "TARGET"]].copy()
reference = identity.loc[identity["SEED"] == 42, ["FOLD", "RID", "TARGET"]]
reference = reference.sort_values(["FOLD", "RID"]).reset_index(drop=True)

for seed in (17, 73):
    candidate = identity.loc[identity["SEED"] == seed, ["FOLD", "RID", "TARGET"]]
    candidate = candidate.sort_values(["FOLD", "RID"]).reset_index(drop=True)
    if not candidate.equals(reference):
        raise ValueError(
            f"Seed {seed} does not use exactly the same fold/RID/target assignments as seed 42."
        )


atomic_write_csv(run_metadata, TABLE_DIR / "validated_run_inventory.csv")
atomic_write_csv(all_predictions, POOLED_DIR / "all_seed_out_of_fold_predictions.csv")

print("\nValidated run inventory:")
display(run_metadata)
print("\nParticipant counts by seed:")
display(seed_counts)
print("\nAll 15 runs passed identity, completion and cohort checks.")


## 1.3. Recompute pooled performance separately for each seed

Each seed is evaluated on its complete 544-participant out-of-fold prediction set. The table reports the seedwise values and the mean, sample standard deviation and range across the three seeds. Seed is the unit of sensitivity analysis; folds are not incorrectly treated as independent repetitions.


In [ ]:
# ============================================================
# 3. Seedwise pooled performance and across-seed summary
# ============================================================

OUTPUT_COLUMNS = {
    "Hybrid": "FINAL_P_pMCI",
    "3MT-only": "THREE_MT_P_pMCI",
    "TMC-only": "TMC_P_pMCI",
}


def expected_calibration_error(targets, probabilities, bins=10):
    predicted = (probabilities >= 0.5).astype(int)
    confidence = np.where(predicted == 1, probabilities, 1.0 - probabilities)
    correct = (predicted == targets).astype(float)
    edges = np.linspace(0.0, 1.0, bins + 1)
    value = 0.0
    for index in range(bins):
        if index == 0:
            mask = (confidence >= edges[index]) & (confidence <= edges[index + 1])
        else:
            mask = (confidence > edges[index]) & (confidence <= edges[index + 1])
        if mask.any():
            value += mask.mean() * abs(correct[mask].mean() - confidence[mask].mean())
    return float(value)


def calculate_metrics(targets, probabilities):
    targets = np.asarray(targets, dtype=int)
    probabilities = np.asarray(probabilities, dtype=float)
    predictions = (probabilities >= CLASSIFICATION_THRESHOLD).astype(int)
    tn, fp, fn, tp = confusion_matrix(targets, predictions, labels=[0, 1]).ravel()
    clipped = np.clip(probabilities, 1e-7, 1.0 - 1e-7)
    return {
        "ROC_AUC": float(roc_auc_score(targets, probabilities)),
        "AVERAGE_PRECISION": float(average_precision_score(targets, probabilities)),
        "ACCURACY": float(accuracy_score(targets, predictions)),
        "BALANCED_ACCURACY": float(balanced_accuracy_score(targets, predictions)),
        "SENSITIVITY": float(tp / (tp + fn)),
        "SPECIFICITY": float(tn / (tn + fp)),
        "PRECISION": float(precision_score(targets, predictions, zero_division=0)),
        "F1": float(f1_score(targets, predictions, zero_division=0)),
        "BRIER_SCORE": float(brier_score_loss(targets, probabilities)),
        "NEGATIVE_LOG_LIKELIHOOD": float(
            log_loss(targets, np.column_stack([1.0 - clipped, clipped]), labels=[0, 1])
        ),
        "EXPECTED_CALIBRATION_ERROR": expected_calibration_error(targets, probabilities),
    }


metric_rows = []
fold_metric_rows = []

for seed in SEEDS:
    seed_table = all_predictions.loc[all_predictions["SEED"] == seed]
    targets = seed_table["TARGET"].to_numpy(dtype=int)
    for output, probability_column in OUTPUT_COLUMNS.items():
        row = {"SEED": seed, "OUTPUT": output}
        row.update(calculate_metrics(targets, seed_table[probability_column]))
        metric_rows.append(row)

        for fold in OUTER_FOLDS:
            fold_table = seed_table.loc[seed_table["FOLD"] == fold]
            fold_row = {"SEED": seed, "FOLD": fold, "OUTPUT": output}
            fold_row.update(
                calculate_metrics(
                    fold_table["TARGET"].to_numpy(dtype=int),
                    fold_table[probability_column].to_numpy(dtype=float),
                )
            )
            fold_metric_rows.append(fold_row)

seedwise_metrics = pd.DataFrame(metric_rows)
seed_fold_metrics = pd.DataFrame(fold_metric_rows)

METRIC_COLUMNS = [
    "ROC_AUC",
    "AVERAGE_PRECISION",
    "ACCURACY",
    "BALANCED_ACCURACY",
    "SENSITIVITY",
    "SPECIFICITY",
    "PRECISION",
    "F1",
    "BRIER_SCORE",
    "NEGATIVE_LOG_LIKELIHOOD",
    "EXPECTED_CALIBRATION_ERROR",
]

summary_rows = []
for output in OUTPUT_COLUMNS:
    subset = seedwise_metrics.loc[seedwise_metrics["OUTPUT"] == output]
    for metric in METRIC_COLUMNS:
        values = subset[metric].to_numpy(dtype=float)
        summary_rows.append(
            {
                "OUTPUT": output,
                "METRIC": metric,
                "N_SEEDS": len(values),
                "MEAN": float(values.mean()),
                "STANDARD_DEVIATION": float(values.std(ddof=1)),
                "MINIMUM": float(values.min()),
                "MAXIMUM": float(values.max()),
                "MEAN_PLUS_MINUS_SD": f"{values.mean():.4f} ± {values.std(ddof=1):.4f}",
            }
        )

across_seed_summary = pd.DataFrame(summary_rows)

atomic_write_csv(seedwise_metrics, TABLE_DIR / "seedwise_pooled_performance.csv")
atomic_write_csv(seed_fold_metrics, TABLE_DIR / "seed_fold_test_performance.csv")
atomic_write_csv(across_seed_summary, TABLE_DIR / "across_seed_performance_summary.csv")

print("Seedwise pooled performance:")
display(seedwise_metrics.round(6))
print("\nAcross-seed mean ± SD:")
display(
    across_seed_summary.loc[
        across_seed_summary["METRIC"].isin(
            ["ROC_AUC", "AVERAGE_PRECISION", "BALANCED_ACCURACY", "BRIER_SCORE"]
        )
    ].round(6)
)


## 1.4. Check whether the principal conclusions survive every seed

This section reports Hybrid-minus-interaction pathway and Hybrid-minus-evidence pathway differences for each seed. It does not call a tiny numerical difference statistically significant; it answers the narrower stability question directly.


In [ ]:
# ============================================================
# 4. Seedwise pathway differences and conclusion checks
# ============================================================

comparison_rows = []

for seed in SEEDS:
    table = seedwise_metrics.loc[seedwise_metrics["SEED"] == seed].set_index("OUTPUT")
    for comparator in ["3MT-only", "TMC-only"]:
        comparison_rows.append(
            {
                "SEED": seed,
                "COMPARISON": f"Hybrid minus {comparator}",
                "ROC_AUC_DIFFERENCE": float(table.loc["Hybrid", "ROC_AUC"] - table.loc[comparator, "ROC_AUC"]),
                "AP_DIFFERENCE": float(table.loc["Hybrid", "AVERAGE_PRECISION"] - table.loc[comparator, "AVERAGE_PRECISION"]),
                "BALANCED_ACCURACY_DIFFERENCE": float(table.loc["Hybrid", "BALANCED_ACCURACY"] - table.loc[comparator, "BALANCED_ACCURACY"]),
                "BRIER_IMPROVEMENT": float(table.loc[comparator, "BRIER_SCORE"] - table.loc["Hybrid", "BRIER_SCORE"]),
            }
        )

seedwise_differences = pd.DataFrame(comparison_rows)

stability_summary = (
    seedwise_differences
    .groupby("COMPARISON")
    .agg(
        SEEDS=("SEED", "count"),
        ROC_AUC_MEAN_DIFFERENCE=("ROC_AUC_DIFFERENCE", "mean"),
        ROC_AUC_MINIMUM_DIFFERENCE=("ROC_AUC_DIFFERENCE", "min"),
        ROC_AUC_POSITIVE_IN_ALL_SEEDS=("ROC_AUC_DIFFERENCE", lambda x: bool((x > 0).all())),
        AP_MEAN_DIFFERENCE=("AP_DIFFERENCE", "mean"),
        AP_POSITIVE_IN_ALL_SEEDS=("AP_DIFFERENCE", lambda x: bool((x > 0).all())),
    )
    .reset_index()
)

atomic_write_csv(seedwise_differences, TABLE_DIR / "seedwise_model_differences.csv")
atomic_write_csv(stability_summary, TABLE_DIR / "principal_conclusion_stability.csv")

display(seedwise_differences.round(6))
print("\nStability of the principal comparisons:")
display(stability_summary.round(6))


## 1.5. Participant-level prediction and gate sensitivity

Because the same held-out participant appears once per seed, this section measures how much each participant’s probability, uncertainty and learned interaction pathway weight change across initialisations. It also counts participants whose 0.50-threshold class changes across seeds.


In [ ]:
# ============================================================
# 5. Participant-level variability across seeds
# ============================================================

variability_rows = []

for output, probability_column in OUTPUT_COLUMNS.items():
    pivot = all_predictions.pivot(index="RID", columns="SEED", values=probability_column)
    if pivot.isna().any().any() or set(pivot.columns) != set(SEEDS):
        raise ValueError(f"Incomplete participant-by-seed matrix for {output}.")

    probability_sd = pivot.std(axis=1, ddof=1)
    class_changes = (pivot >= CLASSIFICATION_THRESHOLD).nunique(axis=1) > 1

    variability_rows.append(
        {
            "OUTPUT": output,
            "PARTICIPANTS": len(pivot),
            "MEAN_WITHIN_PARTICIPANT_PROBABILITY_SD": float(probability_sd.mean()),
            "MEDIAN_WITHIN_PARTICIPANT_PROBABILITY_SD": float(probability_sd.median()),
            "Q95_WITHIN_PARTICIPANT_PROBABILITY_SD": float(probability_sd.quantile(0.95)),
            "MAX_WITHIN_PARTICIPANT_PROBABILITY_SD": float(probability_sd.max()),
            "PARTICIPANTS_WITH_CLASS_CHANGE": int(class_changes.sum()),
            "PROPORTION_WITH_CLASS_CHANGE": float(class_changes.mean()),
        }
    )

gate_pivot = all_predictions.pivot(index="RID", columns="SEED", values="W_3MT")
uncertainty_pivot = all_predictions.pivot(index="RID", columns="SEED", values="FINAL_UNCERTAINTY")

gate_uncertainty_summary = pd.DataFrame(
    [
        {
            "QUANTITY": "W_3MT",
            "MEAN_WITHIN_PARTICIPANT_SD": float(gate_pivot.std(axis=1, ddof=1).mean()),
            "MEDIAN_WITHIN_PARTICIPANT_SD": float(gate_pivot.std(axis=1, ddof=1).median()),
            "MAX_WITHIN_PARTICIPANT_SD": float(gate_pivot.std(axis=1, ddof=1).max()),
        },
        {
            "QUANTITY": "FINAL_UNCERTAINTY",
            "MEAN_WITHIN_PARTICIPANT_SD": float(uncertainty_pivot.std(axis=1, ddof=1).mean()),
            "MEDIAN_WITHIN_PARTICIPANT_SD": float(uncertainty_pivot.std(axis=1, ddof=1).median()),
            "MAX_WITHIN_PARTICIPANT_SD": float(uncertainty_pivot.std(axis=1, ddof=1).max()),
        },
    ]
)

participant_variability = pd.DataFrame(variability_rows)

atomic_write_csv(participant_variability, TABLE_DIR / "participant_prediction_seed_variability.csv")
atomic_write_csv(gate_uncertainty_summary, TABLE_DIR / "gate_and_uncertainty_seed_variability.csv")

display(participant_variability.round(6))
print("\nGate and uncertainty variability:")
display(gate_uncertainty_summary.round(6))


## 1.6. Error detection and conflict-aware uncertainty

Original Hybrid uncertainty and exact conflict-aware uncertainty are evaluated for all three controlled seeds.


In [ ]:
# ============================================================
# 6. Seedwise error detection and conflict-aware uncertainty
# ============================================================

error_detection_rows = []

for seed in SEEDS:
    table = all_predictions.loc[all_predictions["SEED"] == seed].copy()
    errors = (
        (table["FINAL_P_pMCI"] >= CLASSIFICATION_THRESHOLD).astype(int)
        != table["TARGET"].astype(int)
    ).astype(int).to_numpy()

    if np.unique(errors).size < 2:
        raise ValueError(f"Error detection is undefined for seed {seed}.")

    score_definitions = {
        "Original Hybrid uncertainty": "FINAL_UNCERTAINTY",
        "Conflict-aware uncertainty": "CONFLICT_AWARE_UNCERTAINTY",
    }

    for score_name, score_column in score_definitions.items():
        scores = table[score_column].to_numpy(dtype=float)
        correct = scores[errors == 0]
        incorrect = scores[errors == 1]
        error_detection_rows.append(
            {
                "SEED": seed,
                "SCORE": score_name,
                "ERRORS": int(errors.sum()),
                "ERROR_PREVALENCE": float(errors.mean()),
                "ERROR_DETECTION_ROC_AUC": float(roc_auc_score(errors, scores)),
                "ERROR_DETECTION_AVERAGE_PRECISION": float(average_precision_score(errors, scores)),
                "MEAN_CORRECT": float(correct.mean()),
                "MEAN_INCORRECT": float(incorrect.mean()),
                "INCORRECT_MINUS_CORRECT": float(incorrect.mean() - correct.mean()),
            }
        )

error_detection = pd.DataFrame(error_detection_rows)

error_summary = (
    error_detection
    .groupby("SCORE")
    .agg(
        N_SEEDS=("SEED", "count"),
        MEAN_ERROR_DETECTION_ROC_AUC=("ERROR_DETECTION_ROC_AUC", "mean"),
        SD_ERROR_DETECTION_ROC_AUC=("ERROR_DETECTION_ROC_AUC", "std"),
        MEAN_ERROR_DETECTION_AP=("ERROR_DETECTION_AVERAGE_PRECISION", "mean"),
        SD_ERROR_DETECTION_AP=("ERROR_DETECTION_AVERAGE_PRECISION", "std"),
        MEAN_INCORRECT_MINUS_CORRECT=("INCORRECT_MINUS_CORRECT", "mean"),
    )
    .reset_index()
)

atomic_write_csv(error_detection, TABLE_DIR / "seedwise_uncertainty_error_detection.csv")
atomic_write_csv(error_summary, TABLE_DIR / "across_seed_uncertainty_error_detection_summary.csv")

display(error_detection.round(6))
print("\nAcross-seed uncertainty summary:")
display(error_summary.round(6))


## 1.7. Produce a compact dissertation-ready table

The final table contains mean ± SD across the three controlled seeds for the principal performance metrics. Conflict-aware error-detection results remain in their own table because they answer a different evaluation question.


In [ ]:
# ============================================================
# 7. Compact thesis-ready performance table
# ============================================================

THESIS_METRICS = [
    "ROC_AUC",
    "AVERAGE_PRECISION",
    "BALANCED_ACCURACY",
    "SENSITIVITY",
    "SPECIFICITY",
    "BRIER_SCORE",
    "EXPECTED_CALIBRATION_ERROR",
]

thesis_table = (
    across_seed_summary.loc[
        across_seed_summary["METRIC"].isin(THESIS_METRICS),
        ["OUTPUT", "METRIC", "MEAN_PLUS_MINUS_SD"],
    ]
    .pivot(index="OUTPUT", columns="METRIC", values="MEAN_PLUS_MINUS_SD")
    .reindex(["Hybrid", "3MT-only", "TMC-only"])
    .reset_index()
)

atomic_write_csv(thesis_table, TABLE_DIR / "thesis_seed_sensitivity_table.csv")

print("Thesis-ready mean ± SD table:")
display(thesis_table)

print("\nSaved outputs:")
for path in sorted(AGGREGATION_ROOT.rglob("*.csv")):
    print(path)
